In [ ]:
# This is incremental load option 1 where trying to upload csv table > create table addting ingestion_date > do incremental load as a bronze table

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable



StatementMeta(, 2b86f9b0-e13e-4ee0-942a-4e5d414c6ad6, 7, Finished, Available, Finished, False)

In [ ]:

df_src = spark.read.format("csv") \
.option("header", True) \
.option("inferSchema", True) \
.load("Files/Employee/Employee.csv")
display(df_src)

StatementMeta(, 2b86f9b0-e13e-4ee0-942a-4e5d414c6ad6, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5f0f1bec-c384-4b52-83b3-8fa86fb43545)

In [ ]:

from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp

table_name = "Incremental_Load.bronze_employee"

if spark.catalog.tableExists(table_name):

    # ✅ MERGE (incremental)
    delta_table = DeltaTable.forName(spark, table_name)

    delta_table.alias("t").merge(
        df_src.alias("s"),
        "t.EmployeeId = s.EmployeeId"
    ).whenMatchedUpdate(
        condition="""
            t.EmployeeName <> s.EmployeeName OR
            t.Region <> s.Region OR
            t.Dept <> s.Dept
        """,
        set={
            "EmployeeName": "s.EmployeeName",
            "Region": "s.Region",
            "Dept": "s.Dept",
            "ingestion_date": "current_timestamp()"
        }
    ).whenNotMatchedInsert(
        values={
            "EmployeeId": "s.EmployeeId",
            "EmployeeName": "s.EmployeeName",
            "Region": "s.Region",
            "Dept": "s.Dept",
            "ingestion_date": "current_timestamp()"
        }
    ).execute()

else:
    # ✅ First time load ONLY
    df_src.withColumn("ingestion_date", current_timestamp()) \
        .write \
        .mode("overwrite") \
        .format("delta") \
        .option("delta.enableChangeDataFeed", "true") \
        .saveAsTable(table_name)



StatementMeta(, 2b86f9b0-e13e-4ee0-942a-4e5d414c6ad6, 9, Finished, Available, Finished, False)

**New logic started